# Vígil.ia — RF-DETR Small: candidato a backbone de borda (Jetson)

**Missão:** testar o RF-DETR Small (Roboflow — família DETR com backbone DINOv2,
desenhado pra tempo real em edge) pelo MESMO caminho do campeão, e decidir se
ele entra na infraestrutura local (Jetson) do projeto.

**Método (idêntico ao `tira_teima_capacidade.ipynb` — sem vantagem pra ninguém):**
1. **Base** — 12.528 imagens do dataset Roboflow com pseudo-rótulo Otsu (COCO → soja).
2. **Fine-tune 1** — fotos reais do celular (1 grão/foto, pseudo-rótulo saturação + fallback Otsu).
3. **Fine-tune 2** — dataset v3: fotos reais balanceadas + motion blur + cenas multi-grão sintéticas.
4. **Vídeo** — `soja_teste.mp4` (ou `teste_soja.mp4`) com o protocolo padrão:
   NMS agnóstico + ByteTrack + **veredito travado** (≥8 frames, ≥60% de consenso).

**O que decide a vaga no Jetson (seção final):** qualidade no vídeo comparável
ao campeão (YOLO11x_v3, ~95% visual) + latência viável em TensorRT FP16 +
export ONNX limpo.

Pré-requisitos no Drive: dataset 12,5k, fotos reais, o vídeo de teste e
(opcional, pro A/B) `soja_yolo11x_v3.pt`. Tempo estimado na A100: ~6–9 h
(o backbone DINOv2 treina mais devagar por época que a família YOLO).

> O rfdetr treina com dataset **COCO** (formato de export do Roboflow). A
> célula 5 converte os datasets YOLO gerados pelos builders de sempre — o dado
> é bit a bit o mesmo dos outros modelos (seed 42), só muda o formato.


## 1. Setup

In [ ]:
!pip -q install -U rfdetr supervision
!pip -q install "ultralytics==8.4.80"   # só p/ o A/B opcional com o campeão 11x

import os
import torch

try:
    from rfdetr import RFDETRSmall
except ImportError as e:
    raise SystemExit('rfdetr sem RFDETRSmall — atualize: pip install -U rfdetr (>=1.2, '
                     'release nano/small/medium de jul/2025)') from e
import rfdetr
import supervision as sv
print('rfdetr', rfdetr.__version__, '| supervision', sv.__version__)
assert torch.cuda.is_available(), 'Sem GPU! Troque o ambiente de execução.'
print('GPU:', torch.cuda.get_device_name(0))


def best_ckpt(out_dir):
    '''Melhor checkpoint salvo pelo rfdetr (EMA primeiro — costuma ser o melhor).'''
    for n in ('checkpoint_best_ema.pth', 'checkpoint_best_regular.pth',
              'checkpoint_best_total.pth', 'checkpoint.pth'):
        p = os.path.join(out_dir, n)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'nenhum checkpoint em {out_dir}')


def load_small(ckpt):
    '''Carrega um RFDETRSmall a partir de checkpoint nosso. O num_classes fica
    gravado no checkpoint; os fallbacks cobrem versões do pacote que exigem o
    argumento explícito (ids COCO 1..5 -> espaço de 5 ou 6 classes).'''
    err = None
    for kw in ({}, {'num_classes': 5}, {'num_classes': 6}):
        try:
            return RFDETRSmall(pretrain_weights=str(ckpt), **kw)
        except Exception as e:
            err = e
    raise err


## 2. Caminhos e config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'

# Dataset de classificação 12,5k (estágio base)
CLS_BASE = f'{DRIVE}/SoyaBeans Classifications.v2i.folder'
assert os.path.isdir(CLS_BASE), f'CLS_BASE não existe: {CLS_BASE}'

# Fotos reais (fine-tunes): pastas com nomes PT/EN das classes (busca recursiva)
REAL_SRCS = [
    f'{DRIVE}/Soja total/Soja total/Lotes',
    f'{DRIVE}/Soja pra completar',
]

# Vídeo de teste — aceita os dois nomes que já circularam no projeto
VIDEO_TESTE = next((p for p in (f'{DRIVE}/soja_teste.mp4', f'{DRIVE}/teste_soja.mp4')
                    if os.path.exists(p)), None)
assert VIDEO_TESTE, 'Vídeo não encontrado no Drive (soja_teste.mp4 / teste_soja.mp4)!'
print('vídeo de teste:', VIDEO_TESTE)

CHAMP_PT = f'{DRIVE}/soja_yolo11x_v3.pt'   # opcional: campeão p/ A/B no vídeo

EPOCHS_BASE, EPOCHS_FT = 50, 60            # mesmas épocas dos outros modelos

# batch efetivo 16 (recomendação do rfdetr): A100 = 16x1; T4 = 4x4
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH, ACCUM = (16, 1) if vram > 30 else (4, 4)
print(f'VRAM {vram:.0f} GB -> batch_size={BATCH} grad_accum_steps={ACCUM}')


## 3. Builders dos datasets (mesmo código do tira-teima, seed 42)
Pseudo-rótulo Otsu no 12,5k; saturação+Otsu nas fotos reais (segmentando a
imagem ORIGINAL — a correção do ft_real2); v3 = balanceamento + motion blur +
cenas multi-grão sintéticas. Tudo regravado em disco local (treinar lendo do
Drive é lento demais).

In [ ]:
import glob
import hashlib
import unicodedata

import cv2
import numpy as np
import yaml

NAMES = ['broken', 'immature', 'intact', 'skin-damaged', 'spotted']
ALIASES = {0: ['broken', 'quebrad'], 1: ['immature', 'imatur', 'nao maduro'],
           2: ['intact'], 3: ['skin', 'casca', 'ardid', 'danific'], 4: ['spotted', 'manchad']}
IGNORE = ['part of the original']
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
SPLIT_MAP = {'train': 'train', 'valid': 'val', 'val': 'val', 'test': 'test'}
RNG = np.random.default_rng(42)


def norm(s):
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode().lower()


def class_of(folder):
    n = norm(folder)
    if any(norm(k) in n for k in IGNORE):
        return None
    for idx in range(5):
        if any(norm(k) in n for k in ALIASES[idx]):
            return idx
    return None


def otsu_box(img):
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    if area < 0.005 * h * w or area > 0.995 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)


def sat_box(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    s = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    h, w = img.shape[:2]
    if area < 0.01 * h * w or area > 0.90 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)


def letterbox640(img, size=640):
    h, w = img.shape[:2]
    s = size / max(h, w)
    img = cv2.resize(img, (max(1, round(w * s)), max(1, round(h * s))))
    h, w = img.shape[:2]
    top, left = (size - h) // 2, (size - w) // 2
    img = cv2.copyMakeBorder(img, top, size - h - top, left, size - w - left,
                             cv2.BORDER_CONSTANT, value=(0, 0, 0))
    return img, s, left, top


def motion_blur(img, rng=RNG):
    k = int(rng.choice([7, 9, 11, 13, 15]))
    kernel = np.zeros((k, k), np.float32)
    kernel[k // 2, :] = 1.0
    M = cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), float(rng.uniform(0, 180)), 1)
    kernel = cv2.warpAffine(kernel, M, (k, k))
    kernel /= max(kernel.sum(), 1e-6)
    return cv2.filter2D(img, -1, kernel)


def extract_cutout(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    s = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    h, w = img.shape[:2]
    if area < 0.01 * h * w or area > 0.90 * h * w:
        return None
    mask = np.zeros((h, w), np.uint8)
    cv2.drawContours(mask, [c], -1, 255, -1)
    x, y, bw, bh = cv2.boundingRect(c)
    return img[y:y + bh, x:x + bw], mask[y:y + bh, x:x + bw]


def make_scene(cutouts, rng=RNG, size=640):
    bg = int(rng.integers(20, 130))
    canvas = np.clip(np.full((size, size, 3), bg, np.int16)
                     + rng.normal(0, 6, (size, size, 3)), 0, 255).astype(np.uint8)
    occ = np.zeros((size, size), np.uint8)
    boxes = []
    for _ in range(int(rng.integers(6, 26))):
        cls, crop, mask = cutouts[int(rng.integers(len(cutouts)))]
        s = int(rng.integers(60, 150)) / max(crop.shape[:2])
        crop2 = cv2.resize(crop, None, fx=s, fy=s)
        mask2 = cv2.resize(mask, None, fx=s, fy=s, interpolation=cv2.INTER_NEAREST)
        h2, w2 = crop2.shape[:2]
        diag = int(np.ceil(np.hypot(h2, w2))) + 2
        M = cv2.getRotationMatrix2D((w2 / 2, h2 / 2), float(rng.uniform(0, 360)), 1)
        M[0, 2] += (diag - w2) / 2
        M[1, 2] += (diag - h2) / 2
        crop3 = cv2.warpAffine(crop2, M, (diag, diag))
        mask3 = cv2.warpAffine(mask2, M, (diag, diag), flags=cv2.INTER_NEAREST)
        ys, xs = np.where(mask3 > 0)
        if not len(xs):
            continue
        crop3 = crop3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        mask3 = mask3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        gh, gw = mask3.shape
        if gh >= size - 2 or gw >= size - 2:
            continue
        placed = False
        for _try in range(20):
            px = int(rng.integers(0, size - gw))
            py = int(rng.integers(0, size - gh))
            inter = (occ[py:py + gh, px:px + gw] > 0) & (mask3 > 0)
            if inter.sum() <= 0.15 * (mask3 > 0).sum():
                placed = True
                break
        if not placed:
            continue
        alpha = (cv2.GaussianBlur(mask3, (5, 5), 0).astype(np.float32) / 255)[..., None]
        reg = canvas[py:py + gh, px:px + gw]
        canvas[py:py + gh, px:px + gw] = (alpha * crop3 + (1 - alpha) * reg).astype(np.uint8)
        occ[py:py + gh, px:px + gw][mask3 > 0] = 255
        boxes.append((cls, (px + gw / 2) / size, (py + gh / 2) / size, gw / size, gh / size))
    return canvas, boxes


def balance_train(items):
    from collections import defaultdict
    train = [it for it in items if it[2] == 'train']
    rest = [it for it in items if it[2] != 'train']
    by = defaultdict(list)
    for it in train:
        by[it[1]].append(it)
    mx = max(len(v) for v in by.values())
    out = []
    for c, v in by.items():
        out += v + [v[int(i)] for i in RNG.integers(0, len(v), mx - len(v))]
    print('balanceado (train):', {NAMES[c]: sum(1 for it in out if it[1] == c) for c in sorted(by)})
    return out + rest


def collect_base(base_dir):
    '''Acha train/valid/test em qualquer profundidade dentro do dataset 12,5k.'''
    items = []
    for root, dirs, _ in os.walk(base_dir):
        for d in list(dirs):
            sp = SPLIT_MAP.get(d.lower())
            if sp is None:
                continue
            split_dir = os.path.join(root, d)
            for folder in sorted(os.listdir(split_dir)):
                cls = class_of(folder)
                if cls is None:
                    continue
                for p_ in glob.glob(os.path.join(split_dir, folder, '*')):
                    if p_.lower().endswith(IMG_EXT):
                        items.append((p_, cls, sp))
            dirs.remove(d)
    from collections import Counter
    print('coletado (base 12,5k):', dict(Counter(sp for _, _, sp in items)))
    return items


def collect_real(srcs, val_frac=0.15):
    items = []
    for src in srcs:
        for root, _, files in os.walk(src):
            cls = None
            for part in reversed(root.split(os.sep)):
                c = class_of(part)
                if c is not None:
                    cls = c
                    break
            if cls is None:
                continue
            for fn in files:
                if fn.lower().endswith(IMG_EXT):
                    p = os.path.join(root, fn)
                    h = int(hashlib.md5(p.encode()).hexdigest(), 16)
                    items.append((p, cls, 'val' if (h % 100) < val_frac * 100 else 'train'))
    from collections import Counter
    print('coletado (fotos reais):', dict(Counter(sp for _, _, sp in items)))
    return items


def _write_yolo(out_dir, sp, stem, img, lines):
    cv2.imwrite(f'{out_dir}/images/{sp}/{stem}.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    open(f'{out_dir}/labels/{sp}/{stem}.txt', 'w').write(lines)


def build_base(items, out_dir):
    '''Base de detecção: pseudo-rótulo Otsu (1 grão/img, fundo preto) — idêntico
    ao estágio base do RT-DETR e do tira-teima.'''
    assert items, 'Nenhuma imagem do 12,5k coletada!'
    for sp in ('train', 'val', 'test'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)
    kept = skipped = 0
    for i, (path, cls, sp) in enumerate(items):
        if i % 1000 == 0:
            print(f'  base {i}/{len(items)}…', flush=True)
        img = cv2.imread(path)
        if img is None:
            skipped += 1
            continue
        h0, w0 = img.shape[:2]
        box = otsu_box(img)   # na imagem ORIGINAL, nunca depois do letterbox
        if box is None:
            skipped += 1
            continue
        lb, s, left, top = letterbox640(img)
        cx, cy, ww, hh = box
        cx = (cx * w0 * s + left) / 640.0
        cy = (cy * h0 * s + top) / 640.0
        ww = (ww * w0 * s) / 640.0
        hh = (hh * h0 * s) / 640.0
        _write_yolo(out_dir, sp, f'{sp}_{i:06d}', lb,
                    f'{cls} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}')
        kept += 1
    print(f'base: kept={kept} skipped={skipped}')
    assert kept > 0, 'Nenhuma caixa gerada — confira as imagens.'
    yaml.safe_dump({'path': out_dir, 'train': 'images/train', 'val': 'images/val',
                    'test': 'images/test', 'names': {i: n for i, n in enumerate(NAMES)}},
                   open(f'{out_dir}/data.yaml', 'w'), sort_keys=False, allow_unicode=True)
    return f'{out_dir}/data.yaml'


def build_real_single(items, out_dir):
    '''FT1: fotos reais, 1 grão/foto — pseudo-rótulo saturação (fundo cinza) com
    fallback Otsu, caixa mapeada pro letterbox 640.'''
    assert items, 'Nenhuma foto real coletada! Confira REAL_SRCS.'
    for sp in ('train', 'val'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)
    kept = skipped = 0
    for i, (path, cls, sp) in enumerate(items):
        if i % 200 == 0:
            print(f'  reais {i}/{len(items)}…', flush=True)
        img = cv2.imread(path)
        if img is None:
            skipped += 1
            continue
        h0, w0 = img.shape[:2]
        box = sat_box(img) or otsu_box(img)
        if box is None:
            skipped += 1
            continue
        lb, s, left, top = letterbox640(img)
        cx, cy, ww, hh = box
        cx = (cx * w0 * s + left) / 640.0
        cy = (cy * h0 * s + top) / 640.0
        ww = (ww * w0 * s) / 640.0
        hh = (hh * h0 * s) / 640.0
        _write_yolo(out_dir, sp, f'{sp}_{i:06d}', lb,
                    f'{cls} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}')
        kept += 1
    print(f'reais: kept={kept} skipped={skipped}')
    assert kept > 0, 'Nenhuma caixa gerada nas fotos reais.'
    yaml.safe_dump({'path': out_dir, 'train': 'images/train', 'val': 'images/val',
                    'names': {i: n for i, n in enumerate(NAMES)}},
                   open(f'{out_dir}/data.yaml', 'w'), sort_keys=False, allow_unicode=True)
    return f'{out_dir}/data.yaml'


def build_v3(items, out_dir, n_synth=600, blur_frac=0.4):
    '''FT2: dataset v3 — reais balanceadas ×2 com motion blur + cenas multi-grão
    sintéticas (compositor de recortes), 40% das cenas borradas.'''
    assert items, 'Nenhuma imagem coletada! Confira REAL_SRCS.'
    for sp in ('train', 'val', 'test'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)
    items = balance_train(items)
    cutouts = []
    kept = skipped = 0
    for i, (path, cls, sp) in enumerate(items):
        if i % 200 == 0:
            print(f'  fotos {i}/{len(items)}…', flush=True)
        img = cv2.imread(path)
        if img is None:
            skipped += 1
            continue
        h0, w0 = img.shape[:2]
        box = sat_box(img) or otsu_box(img)
        if box is None:
            skipped += 1
            continue
        if sp == 'train':
            cut = extract_cutout(img)
            if cut is not None:
                cutouts.append((cls, cut[0], cut[1]))
        lb, s, left, top = letterbox640(img)
        cx, cy, ww, hh = box
        cx = (cx * w0 * s + left) / 640.0
        cy = (cy * h0 * s + top) / 640.0
        ww = (ww * w0 * s) / 640.0
        hh = (hh * h0 * s) / 640.0
        line = f'{cls} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}'
        stem = f'{sp}_{i:06d}'
        _write_yolo(out_dir, sp, stem, lb, line)
        kept += 1
        if sp == 'train':
            _write_yolo(out_dir, 'train', stem + 'b', motion_blur(lb), line)
            kept += 1
    print(f'fotos reais: kept={kept} skipped={skipped} | recortes: {len(cutouts)}')
    assert cutouts, 'Nenhum recorte extraído!'
    synth = 0
    for j in range(n_synth):
        if j % 100 == 0:
            print(f'  cenas {j}/{n_synth}…', flush=True)
        canvas, boxes = make_scene(cutouts)
        if not boxes:
            continue
        if RNG.random() < blur_frac:
            canvas = motion_blur(canvas)
        _write_yolo(out_dir, 'train', f'synth_{j:05d}', canvas,
                    '\n'.join(f'{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}'
                              for c, cx, cy, w, h in boxes))
        synth += 1
    print(f'cenas sintéticas: {synth}')
    yaml.safe_dump({'path': out_dir, 'train': 'images/train', 'val': 'images/val',
                    'test': 'images/test', 'names': {i: n for i, n in enumerate(NAMES)}},
                   open(f'{out_dir}/data.yaml', 'w'), sort_keys=False, allow_unicode=True)
    return f'{out_dir}/data.yaml'


## 4. Constrói os 3 datasets (YOLO-format, disco local)

In [ ]:
BASE_DIR = '/content/soja_det_base'
REAL_DIR = '/content/soja_det_real'
V3_DIR = '/content/soja_det_v3'

if not os.path.exists(f'{BASE_DIR}/data.yaml'):
    build_base(collect_base(CLS_BASE), BASE_DIR)
real_items = collect_real(REAL_SRCS)
if not os.path.exists(f'{REAL_DIR}/data.yaml'):
    build_real_single(real_items, REAL_DIR)
if not os.path.exists(f'{V3_DIR}/data.yaml'):
    build_v3(real_items, V3_DIR)
print('datasets YOLO prontos ✅')


## 5. Converte YOLO → COCO (formato que o rfdetr consome)
Cada split vira `train/ valid/ test/` com `_annotations.coco.json` (estilo
export do Roboflow, categorias 1..5). Mesmas imagens, mesmas caixas.

In [ ]:
import json
import shutil


def yolo_to_coco(yolo_dir, out_dir):
    cats = [{'id': i + 1, 'name': n, 'supercategory': 'soybeans'}
            for i, n in enumerate(NAMES)]
    for sp_src, sp_dst in (('train', 'train'), ('val', 'valid'), ('test', 'test')):
        files = sorted(glob.glob(f'{yolo_dir}/images/{sp_src}/*.jpg'))
        if not files:
            continue
        chk = cv2.imread(files[0])
        assert chk.shape[:2] == (640, 640), f'esperava 640x640, veio {chk.shape[:2]}'
        dst = f'{out_dir}/{sp_dst}'
        os.makedirs(dst, exist_ok=True)
        images, anns, aid = [], [], 1
        for iid, p in enumerate(files, 1):
            fn = os.path.basename(p)
            if not os.path.exists(f'{dst}/{fn}'):
                shutil.copy(p, f'{dst}/{fn}')
            images.append({'id': iid, 'file_name': fn, 'width': 640, 'height': 640})
            lp = f'{yolo_dir}/labels/{sp_src}/{fn[:-4]}.txt'
            if os.path.exists(lp):
                for line in open(lp).read().splitlines():
                    if not line.strip():
                        continue
                    c, cx, cy, ww, hh = line.split()
                    cx, cy, ww, hh = (float(v) * 640 for v in (cx, cy, ww, hh))
                    anns.append({'id': aid, 'image_id': iid, 'category_id': int(c) + 1,
                                 'bbox': [round(cx - ww / 2, 2), round(cy - hh / 2, 2),
                                          round(ww, 2), round(hh, 2)],
                                 'area': round(ww * hh, 2), 'iscrowd': 0, 'segmentation': []})
                    aid += 1
        json.dump({'info': {'description': f'Vigil.ia {os.path.basename(out_dir)} {sp_dst}'},
                   'licenses': [], 'categories': cats, 'images': images, 'annotations': anns},
                  open(f'{dst}/_annotations.coco.json', 'w'))
        print(f'{out_dir}/{sp_dst}: {len(images)} imgs, {len(anns)} caixas')
    return out_dir


RF_BASE = yolo_to_coco(BASE_DIR, '/content/rf_base')
RF_REAL = yolo_to_coco(REAL_DIR, '/content/rf_real')
RF_V3 = yolo_to_coco(V3_DIR, '/content/rf_v3')

ID2NAME = {i + 1: n for i, n in enumerate(NAMES)}   # category_id COCO -> nome
print('classes:', ID2NAME)


In [ ]:
# Sanidade visual: caixas COCO certas ANTES de gastar GPU
import matplotlib.pyplot as plt

jd = json.load(open('/content/rf_base/train/_annotations.coco.json'))
by_img = {}
for a in jd['annotations']:
    by_img.setdefault(a['image_id'], []).append(a)
plt.figure(figsize=(12, 7))
for k, im in enumerate(jd['images'][:6]):
    img = cv2.cvtColor(cv2.imread(f"/content/rf_base/train/{im['file_name']}"),
                       cv2.COLOR_BGR2RGB)
    titulo = []
    for a in by_img.get(im['id'], []):
        x, y, w, h = [int(v) for v in a['bbox']]
        cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 3)
        titulo.append(ID2NAME[a['category_id']])
    ax = plt.subplot(2, 3, k + 1)
    ax.imshow(img)
    ax.set_title(','.join(titulo))
    ax.axis('off')
plt.tight_layout()
plt.show()


## 6. Estágio 1 — RF-DETR Small base (12,5k)
Pré-treino COCO do pacote → transfer learning na soja. As augmentations aqui são
as internas do rfdetr (não dá pra passar os knobs do ultralytics) — o que mais
importa pro nosso domínio (blur, multi-grão, fundo variado) entra pelo DADO nos
fine-tunes, então a comparação segue honesta. A resolução fica a padrão do
Small (as imagens em disco estão em 640; ele redimensiona internamente — se
quiser fixar `resolution=`, o pacote valida o múltiplo exigido).

In [ ]:
OUT_BASE = '/content/runs_rfdetr/small_base12k'
os.makedirs(OUT_BASE, exist_ok=True)

base = RFDETRSmall()   # pesos COCO padrão do pacote
base.train(dataset_dir=RF_BASE, epochs=EPOCHS_BASE, batch_size=BATCH,
           grad_accum_steps=ACCUM, lr=1e-4, output_dir=OUT_BASE)

BASE_BEST = best_ckpt(OUT_BASE)
!cp {BASE_BEST} {DRIVE}/soja_rfdetr_small_base12k.pth
del base
torch.cuda.empty_cache()
print('backup no Drive: soja_rfdetr_small_base12k.pth <-', BASE_BEST)


In [ ]:
# Avaliação do base no test do 12,5k: acurácia top-1 (1 grão/img) + matriz
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns


def eval_single_grain(model, rf_dir, split, thr=0.30, title=''):
    jd = json.load(open(f'{rf_dir}/{split}/_annotations.coco.json'))
    id2name = {c['id']: c['name'] for c in jd['categories']}
    by_img = {}
    for a in jd['annotations']:
        by_img.setdefault(a['image_id'], []).append(a)
    true, pred = [], []
    for k, im in enumerate(jd['images']):
        anns = by_img.get(im['id'], [])
        if len(anns) != 1:
            continue
        if k % 200 == 0:
            print(f'  {k}/{len(jd["images"])}…', flush=True)
        det = model.predict(
            Image.open(f"{rf_dir}/{split}/{im['file_name']}").convert('RGB'),
            threshold=thr)
        if len(det) == 0:
            p = 'nada'
        else:
            p = id2name.get(int(det.class_id[int(np.argmax(det.confidence))]), '?')
        true.append(id2name[anns[0]['category_id']])
        pred.append(p)
    acc = float(np.mean([t == p for t, p in zip(true, pred)]))
    print(f'{title} acc = {acc:.1%}  ({len(true)} imagens)')
    return acc, true, pred


base_eval = load_small(BASE_BEST)
acc_base, t, p = eval_single_grain(base_eval, RF_BASE, 'test', title='BASE 12,5k (test):')
labels = NAMES + ['nada']
print(classification_report(t, p, labels=labels, zero_division=0))
cm = confusion_matrix(t, p, labels=labels)
plt.figure(figsize=(7, 5.5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels,
            cmap='Greens', linewidths=0.5)
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title(f'RF-DETR Small base — test {acc_base:.0%}')
plt.tight_layout()
plt.show()
del base_eval
torch.cuda.empty_cache()


## 7. Fine-tune 1 — fotos reais (domínio do celular)
lr menor (5e-5) pra não esquecer o base. Baseline antes vs depois no val real —
é o número do domain shift.

In [ ]:
antes = load_small(BASE_BEST)
acc_real_antes, _, _ = eval_single_grain(antes, RF_REAL, 'valid',
                                         title='base -> fotos reais (val):')
del antes
torch.cuda.empty_cache()

OUT_FT1 = '/content/runs_rfdetr/small_ft_real'
os.makedirs(OUT_FT1, exist_ok=True)
ft1 = load_small(BASE_BEST)
ft1.train(dataset_dir=RF_REAL, epochs=EPOCHS_FT, batch_size=BATCH,
          grad_accum_steps=ACCUM, lr=5e-5, output_dir=OUT_FT1)
FT1_BEST = best_ckpt(OUT_FT1)
!cp {FT1_BEST} {DRIVE}/soja_rfdetr_small_ft_real.pth
del ft1
torch.cuda.empty_cache()

depois = load_small(FT1_BEST)
acc_real_depois, _, _ = eval_single_grain(depois, RF_REAL, 'valid',
                                          title='FT1 -> fotos reais (val):')
del depois
torch.cuda.empty_cache()
print(f'domínio real: {acc_real_antes:.1%} -> {acc_real_depois:.1%}')


## 8. Fine-tune 2 — dataset v3 (multi-grão + blur + balanceamento)
Parte do melhor do FT1. É o estágio que curou o "tudo vira broken no borrão"
nos outros modelos.

In [ ]:
OUT_FT2 = '/content/runs_rfdetr/small_v3'
os.makedirs(OUT_FT2, exist_ok=True)
ft2 = load_small(FT1_BEST)
ft2.train(dataset_dir=RF_V3, epochs=EPOCHS_FT, batch_size=BATCH,
          grad_accum_steps=ACCUM, lr=5e-5, output_dir=OUT_FT2)
FT2_BEST = best_ckpt(OUT_FT2)
!cp {FT2_BEST} {DRIVE}/soja_rfdetr_small_v3.pth
del ft2
torch.cuda.empty_cache()
print('modelo final do funil: soja_rfdetr_small_v3.pth <-', FT2_BEST)


In [ ]:
# mAP no val do v3 (40 fotos reais) — comparável ao placar do tira-teima
from supervision.metrics import MeanAveragePrecision, MetricTarget

final = load_small(FT2_BEST)
ds = sv.DetectionDataset.from_coco(f'{RF_V3}/valid',
                                   f'{RF_V3}/valid/_annotations.coco.json')
name_to_idx = {n: i for i, n in enumerate(ds.classes)}
metric = MeanAveragePrecision(metric_target=MetricTarget.BOXES)
for path, _, gt in ds:
    det = final.predict(Image.open(path).convert('RGB'), threshold=0.05)
    if len(det):
        det.class_id = np.array([name_to_idx.get(ID2NAME.get(int(c), '?'), 0)
                                 for c in det.class_id])
    metric.update([det], [gt])
map_v3 = metric.compute()
print(map_v3)
try:
    MAP50_V3, MAP5095_V3 = float(map_v3.map50), float(map_v3.map50_95)
except AttributeError:
    MAP50_V3 = MAP5095_V3 = float('nan')


## 9. Vídeo — o juiz de verdade
Mesmo protocolo dos comparativos: NMS **agnóstico de classe** (a família DETR
não roda NMS — duplicata viraria caixa dupla), ByteTrack (via supervision) e
veredito travado (≥8 frames, ≥60% de consenso). Sai `rfdetr_small_teste.mp4`
no Drive pra assistir lado a lado com o `tirateima_yolo11x.mp4`.

In [ ]:
from collections import defaultdict, Counter

LOCK_MIN_FRAMES, LOCK_RATIO = 8, 0.60   # protocolo padrão dos comparativos
COLORS = {'intact': (80, 200, 80), 'immature': (60, 200, 200),
          'broken': (200, 100, 160), 'skin-damaged': (60, 160, 255),
          'spotted': (80, 80, 230)}


def locked_video_rf(model, src, out_path, conf=0.35):
    votes = defaultdict(Counter)
    frames_seen = Counter()
    locked = {}
    cap = cv2.VideoCapture(src)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    tracker = sv.ByteTrack(frame_rate=int(round(fps)))
    writer = None
    n = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        n += 1
        if writer is None:
            h, w = frame.shape[:2]
            writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
        det = model.predict(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), threshold=conf)
        det = det.with_nms(threshold=0.6, class_agnostic=True)
        det = tracker.update_with_detections(det)
        if len(det) and det.tracker_id is not None:
            for (x1, y1, x2, y2), tid, cid, cf in zip(det.xyxy.astype(int), det.tracker_id,
                                                      det.class_id, det.confidence):
                tid = int(tid)
                name = ID2NAME.get(int(cid))
                if name is None:
                    continue
                if tid not in locked:
                    votes[tid][name] += float(cf)
                    frames_seen[tid] += 1
                    top, w_ = votes[tid].most_common(1)[0]
                    if (frames_seen[tid] >= LOCK_MIN_FRAMES
                            and w_ >= LOCK_RATIO * sum(votes[tid].values())):
                        locked[tid] = top
                if tid in locked:
                    cls = locked[tid]
                    color = COLORS[cls]
                    label = f'#{tid} {cls}'
                else:
                    color = (160, 160, 160)
                    label = f'#{tid} analisando…'
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, label, (x1, max(18, y1 - 6)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
        writer.write(frame)
        if n % 100 == 0:
            print(f'  frame {n}…', flush=True)
    cap.release()
    writer.release()
    for tid, cnt in votes.items():
        locked.setdefault(tid, cnt.most_common(1)[0][0])
    intact = sum(1 for c in locked.values() if c == 'intact')
    pct = intact / len(locked) if locked else 0
    print(f'  grãos: {len(locked)} | intactos: {intact} ({pct * 100:.0f}%)'
          f' -> {"APROVADO" if pct >= 0.9 else "REPROVADO"}')
    return locked


try:
    final.optimize_for_inference()
    print('optimize_for_inference ✅')
except Exception as e:
    print('(seguindo sem optimize_for_inference:', e, ')')

VID_OUT = '/content/rfdetr_small_teste.mp4'
verdict_rf = locked_video_rf(final, VIDEO_TESTE, VID_OUT)
!cp {VID_OUT} {DRIVE}/
print('vídeo anotado no Drive: rfdetr_small_teste.mp4')


In [ ]:
# A/B opcional — campeão YOLO11x_v3 no MESMO vídeo/protocolo (lado a lado)
verdict_champ = None
if os.path.exists(CHAMP_PT):
    from ultralytics import YOLO
    champ = YOLO(CHAMP_PT)
    votes = defaultdict(Counter)
    frames_seen = Counter()
    locked = {}
    cap = cv2.VideoCapture(VIDEO_TESTE)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    cap.release()
    writer = None
    out = '/content/tirateima_yolo11x_ref.mp4'
    for r in champ.track(source=VIDEO_TESTE, conf=0.35, iou=0.6, agnostic_nms=True,
                         tracker='bytetrack.yaml', stream=True, persist=True, verbose=False):
        frame = r.orig_img.copy()
        if writer is None:
            h, w = frame.shape[:2]
            writer = cv2.VideoWriter(out, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
        if r.boxes.id is not None:
            for xyxy, tid, c, cf in zip(r.boxes.xyxy.cpu().numpy().astype(int),
                                        r.boxes.id.int().tolist(),
                                        r.boxes.cls.int().tolist(),
                                        r.boxes.conf.tolist()):
                x1, y1, x2, y2 = xyxy
                name = NAMES[c]
                if tid not in locked:
                    votes[tid][name] += cf
                    frames_seen[tid] += 1
                    top, w_ = votes[tid].most_common(1)[0]
                    if (frames_seen[tid] >= LOCK_MIN_FRAMES
                            and w_ >= LOCK_RATIO * sum(votes[tid].values())):
                        locked[tid] = top
                cls = locked.get(tid)
                color = COLORS[cls] if cls else (160, 160, 160)
                label = f'#{tid} {cls or "analisando…"}'
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, label, (x1, max(18, y1 - 6)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
        writer.write(frame)
    writer.release()
    for tid, cnt in votes.items():
        locked.setdefault(tid, cnt.most_common(1)[0][0])
    verdict_champ = locked
    intact = sum(1 for c in locked.values() if c == 'intact')
    print(f'campeão 11x: grãos {len(locked)} | intactos {intact}'
          f' ({100 * intact / max(len(locked), 1):.0f}%)')
    !cp {out} {DRIVE}/
else:
    print('(sem soja_yolo11x_v3.pt no Drive — A/B pulado; '
          'compare com o tirateima_yolo11x.mp4 já gerado no tira-teima)')


## 10. Benchmark + export ONNX (o caminho do Jetson)

In [ ]:
import time

# parâmetros do modelo (contados, não de tabela)
try:
    net = final.model.model
    params_m = sum(p.numel() for p in net.parameters()) / 1e6
    print(f'parâmetros: {params_m:.1f} M')
except Exception as e:
    params_m = float('nan')
    print('(não consegui contar parâmetros:', e, ')')

# latência média em 100 frames do vídeo — referência de GPU Colab, NÃO é número
# de Jetson (lá o que vale é TensorRT FP16, medido no aparelho)
cap = cv2.VideoCapture(VIDEO_TESTE)
frames = []
while len(frames) < 110:
    ok, f = cap.read()
    if not ok:
        break
    frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
cap.release()
for f in frames[:10]:
    final.predict(f, threshold=0.35)
t0 = time.time()
for f in frames[10:]:
    final.predict(f, threshold=0.35)
ms = 1000 * (time.time() - t0) / max(len(frames) - 10, 1)
print(f'latência: {ms:.1f} ms/frame ≈ {1000 / ms:.0f} fps na',
      torch.cuda.get_device_name(0))

# export ONNX -> no Jetson: trtexec --onnx=soja_rfdetr_small_v3.onnx --fp16
try:
    exp = load_small(FT2_BEST)
    exp.export(output_dir='/content/rfdetr_onnx')
    onnxs = glob.glob('/content/rfdetr_onnx/**/*.onnx', recursive=True)
    assert onnxs, 'export rodou mas não achei .onnx'
    !cp {onnxs[0]} {DRIVE}/soja_rfdetr_small_v3.onnx
    print('ONNX no Drive: soja_rfdetr_small_v3.onnx')
    del exp
    torch.cuda.empty_cache()
except Exception as e:
    print('export ONNX falhou:', e)
    print("-> tente: pip install 'rfdetr[onnxexport]' e rode esta célula de novo")


## 11. Placar final

In [ ]:
n_rf = len(verdict_rf)
i_rf = sum(1 for c in verdict_rf.values() if c == 'intact')
print('=== PLACAR — RF-DETR Small (base 12,5k -> FT fotos reais -> FT v3) ===')
print(f'  parâmetros:            {params_m:.1f} M')
print(f'  base 12,5k (test):     {acc_base:.1%}  (acc top-1, 1 grão/img)')
print(f'  fotos reais (val):     {acc_real_antes:.1%} -> {acc_real_depois:.1%}  (antes -> depois do FT1)')
print(f'  v3 (val)  mAP50:       {MAP50_V3:.3f}   mAP50-95: {MAP5095_V3:.3f}')
print(f'  vídeo:                 {n_rf} grãos | {i_rf} intactos ({100 * i_rf / max(n_rf, 1):.0f}%)')
print(f'  latência (Colab GPU):  {ms:.1f} ms/frame')
if verdict_champ:
    n_c = len(verdict_champ)
    i_c = sum(1 for c in verdict_champ.values() if c == 'intact')
    print(f'  campeão 11x (mesmo vídeo): {n_c} grãos | {i_c} intactos'
          f' ({100 * i_c / max(n_c, 1):.0f}%)')
print()
print('Linha p/ colar no COMPARATIVO_YOLO11S_VS_RTDETR.md:')
print(f'| RF-DETR Small | {params_m:.1f} M | base {acc_base:.1%} | '
      f'real {acc_real_antes:.0%}->{acc_real_depois:.0%} | mAP50 v3 {MAP50_V3:.3f} | '
      f'vídeo {n_rf} grãos / {100 * i_rf / max(n_rf, 1):.0f}% intact | '
      f'{ms:.1f} ms/frame (Colab) |')


## 12. Como ler o resultado — a vaga no Jetson

O juiz continua sendo o vídeo: assista `rfdetr_small_teste.mp4` lado a lado com
o `tirateima_yolo11x.mp4` e estime a acurácia premium, como das outras vezes.

| Critério | Bar mínimo pro Jetson |
|---|---|
| Qualidade no vídeo (visual) | ≥ candidato local atual (YOLO11s/11m destilado); idealmente ≈ campeão 11x (~95%) |
| Latência | tempo real no Orin com **TensorRT FP16** — usar o ONNX exportado aqui (`trtexec --onnx=soja_rfdetr_small_v3.onnx --fp16`) e medir NO aparelho; o número do Colab é só referência |
| Pipeline | ponto a favor do DETR: **sem NMS** no pós-processamento (menos latência e menos código no edge) |
| Treino | anotar qualquer instabilidade (a família DETR já nos deu colapso de warmup no RT-DETR; o rfdetr tem receita própria de treino) |

Cenários:
- **RF-DETR Small ≈ 11x no vídeo** → candidato fortíssimo pro Jetson (qualidade
  de transformer em tamanho de edge); vira o alvo da infraestrutura local e
  pode até dispensar a destilação 11x → 11s/11m.
- **RF-DETR Small ≈ 11s (abaixo do 11x)** → empata com o plano atual; quem
  decide é a latência medida no Jetson e a simplicidade de deploy.
- **Abaixo do 11s** → registrar como resultado negativo (como o LR
  discriminativo) e seguir com a destilação.

Em qualquer cenário: atualizar o `COMPARATIVO_YOLO11S_VS_RTDETR.md` com a
rodada (linha pronta na célula do placar) e guardar os 3 checkpoints do Drive
(`soja_rfdetr_small_base12k.pth`, `soja_rfdetr_small_ft_real.pth`,
`soja_rfdetr_small_v3.pth`).
